# Processing Workflows

While the descriptions from the first notebook are nice, they're not research data or archival metadata.

This next section details a workflow that will process your materials into a structured format that can be used for research or library systems. This is an example that can be adapted to your specific needs.

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1-wksvuqBUv8qq2mLJ8EbbluStjyvXIp7#scrollTo=OsonjvCPSjIr)  

# Sending your files to the model 

For many models, you just need to provide an image's URL on the Internet.  The model will fetch the image and encode it for you. However, for local files on your computer, you'll need to send your data to the model for processing. 

The most common way to do this is to encode the binary data in your file as a string of characters using the base64 encoding system.  So rather than sending ones and zeros over the Internet, we'll be sending packets of text characters.  On the other end, the model turns that data back into an image file to process.

In [1]:
import base64
from pathlib import Path 

def encode_images(image_directory: str, file_extensions: list = ['.jpg', '.png'], data_uri: bool = True):
    """Generator to encode images in the specified directory to base64 strings.
    
    Args:
        image_directory (Path): The directory containing images to encode.
        file_extensions (list, optional): List of file extensions to consider. Defaults to ['.jpg', '.png'].
        data_uri (bool, optional): If True, includes the data URI scheme in the output. Defaults to True.

    Yields:
        str: Base64 encoded string of each image.
    """
    image_directory = Path(image_directory)
    if not image_directory.is_dir():
        raise ValueError(f"The provided path {image_directory} is not a valid directory.")
    image_paths = [p for p in image_directory.iterdir() if p.suffix.lower() in file_extensions]
    for path in image_paths:
        with open(path, 'rb') as img_file:
            encoded_string = base64.b64encode(img_file.read()).decode('utf-8')
            if data_uri:
                mime_type = 'image/jpeg' if path.suffix.lower() == '.jpg' else 'image/png'
                yield path, f"data:{mime_type};base64,{encoded_string}"
            else:
                yield path, encoded_string




In [2]:
base_url = "https://glw30r2npdvyvzho.us-east-1.aws.endpoints.huggingface.cloud/v1"
api_key = "not needed for this workshop"  
model = "Qwen/Qwen3-VL-8B-Instruct"  # Qwen3-VL-8B-Instruct

In [ ]:
base_url = "https://api.parasail.io/v1"
api_key = "psk-"  # Replace with your actual Parasail API key
model = "Qwen/Qwen3-VL-8B-Instruct"

In [3]:
import httpx
img = "https://images.eap.bl.uk/EAP1477/EAP1477_1_1_1/350.jp2/full/full/0/default.jpg"
images = encode_images("img")
img = next(images)[1]

url = f"{base_url}/chat/completions"
headers = {"Authorization": f"Bearer {api_key}", "content-type": "application/json"}
prompt = "Extract and return all the text from this image as markdown. Include all text elements and maintain the reading order."
data = {
    "model": model,
    "messages": [
        {   "role": "system", 
            "content": "You are a helpful assistant that extracts text from images using OCR."
        },
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": img}},
                {"type": "text", "text": prompt},
            ],
        },
    ],
}
response = httpx.post(url, headers=headers, json=data, timeout=60)
response.json()

KeyboardInterrupt: 

In [3]:
from IPython.display import Image as IPythonImage

display(IPythonImage(graph.get_graph().draw_mermaid_png()))

NameError: name 'graph' is not defined

In [ ]:
import os
import asyncio
import aiohttp
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm

# Use AsyncOpenAI instead of OpenAI
client = AsyncOpenAI(
    api_key=api_key,
    base_url=base_url,
)

async def process_image(img: tuple, session: aiohttp.ClientSession, semaphore: asyncio.Semaphore):
    """Process a single image with rate limiting and timeout"""
    async with semaphore:  # Limit concurrent requests
        image_path = img[0]
        image_b64 = img[1]
        file = {
            "name": image_path.name,
        }
        #print(f"Starting to process: {image_path.name}")

        try:
            # Add timeout to API call as well
            completion = await asyncio.wait_for(
                client.chat.completions.create(
                    model=model,
                    messages=[
                        {
                            "role": "user",
                            "content": [
                                {
                                    "type": "text",
                                    "text": "Extract text from the image. Preserve reading order. Return text as markdown."
                                },
                                {
                                    "type": "image_url",
                                    "image_url": {
                                        "url": image_b64
                                    }
                                }
                            ]
                        }
                    ],
                ),
                timeout=360  # 360 second timeout for API call
            )
            #print(f"Successfully processed: {image_path.name}")
            file["markdown_text"] = completion.choices[0].message.content
            return file
        except asyncio.TimeoutError:
            print(f"API timeout for image {image_path.name}")
            return None
        except Exception as e:
            print(f"Error processing image {image_path.name}: {e}")
            return None

async def process_all_images(images, max_concurrent=3):
    """Process all images concurrently with a limit on concurrent requests"""
    # Create aiohttp session for connection pooling
    connector = aiohttp.TCPConnector(limit=10, limit_per_host=5)
    timeout = aiohttp.ClientTimeout(total=300)  # 5 minute total timeout
    
    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        semaphore = asyncio.Semaphore(max_concurrent)
        
        # Create tasks for all images
        tasks = [process_image(image, session, semaphore) for image in images]
        
        # Process with progress bar
        results = []
        completed_count = 0
        
        for task in tqdm.as_completed(tasks, total=len(tasks), desc="Processing images"):
            try:
                result = await task
                completed_count += 1
                if result is not None:
                    results.append(result)
                #print(f"Completed {completed_count}/{len(tasks)} images")
            except Exception as e:
                print(f"Task failed with error: {e}")
                completed_count += 1
    
    return results



# State

As we move from step to step in the workflow, we will need something to store the data from prior steps and to make that data available. This object is often called "state" in programming. In the cell below, we create a simple state object to hold our data as we move through the workflow.

Given that we're processing a box of archival materials, I have called it "Box." I find it helpful to think of the fields in the object as columns in a spreadsheet. Each field (box_manifest_uri, image_uris...) gives us a placeholder for information that we'd like to gather during the workflow. How you design your state object can be helpful in deciding what steps are needed in the workflow and what your final output should look like.

In [5]:
# https://langchain-ai.github.io/langgraph/tutorials/workflows/#prompt-chaining
# https://typing.python.org/en/latest/spec/typeddict.html

from typing_extensions import TypedDict


class Box(TypedDict):  # Level: File in EAP
    box_manifest_uri: str
    box_manifest_data: dict
    box_metadata: dict
    box_summary: str
    image_urls: list[str]
    image_height: int
    image_width: int
    images: list[dict]
    item_grouping: list
    items: dict
    image_descriptions: bool
    start_index: int  # Start processing from this image index
    end_index: int  # End processing at this image index
    file_lookup: dict

# Steps in the Workflow

In [6]:
import re
import json
import requests
import srsly
from rich import print
from tqdm.asyncio import tqdm
from pathlib import Path
import nest_asyncio
from openai import OpenAI



# helper functions for the cleaning step
def remove_code_tags(text):
    """Remove code tags (triple backticks) from the text. Also remove tags with language specifiers such as ```yaml."""
    return re.sub(r"```[a-zA-Z]*\n(.*?)\n```", r"\1", text, flags=re.DOTALL)

def remove_repeated_phrases(text):
    """Remove repeated phrases in the text."""
    # This regex looks for any phrase (sequence of words) that is repeated consecutively
    pattern = r"(\b\w+(?: \w+){0,5}\b)( \1)+"
    return re.sub(pattern, r"\1", text)

# https://docs.langchain.com/oss/python/langgraph/workflows-agents


# Steps
def load_manifest(state: Box) -> Box:
    if Path('img/info.json').exists():
        info = srsly.read_json('img/info.json')
        manifest_url = info.get('url', None)
        header = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/11"
        }
        manifest = requests.get(manifest_url, headers=header)
        if manifest.status_code != 200:
            raise Exception(f"Error downloading manifest: {manifest.status_code}")

        manifest = manifest.json()
        metadata = {}
        for item in manifest["metadata"]:
            metadata[item["label"]] = item["value"]
        print(f"Loaded manifest for {metadata.get('Identifier', 'unknown_box')}")
        file_lookup = info['images']
        return {
            "box_metadata": metadata,
            "file_lookup": file_lookup
        }
    else:
        print("info.json not found in img/ directory. Please retry downloading the files")


def text_recognition(state: Box) -> Box:
    """Extract text from images using OCR (synchronous wrapper).

    This step prefers to use an already-produced `output` variable (from a prior
    async cell). If `output` isn't available it will attempt to run the
    async processing, applying nest_asyncio when running inside a Jupyter
    event loop. If nest_asyncio isn't available, instruct the user to run the
    async cell to produce `output`.
    """
    from tqdm import tqdm
    
    # If the async cell has already produced `output`, reuse it (fast and safe).
    if "output" in globals() and isinstance(output, list):
        print("✅ Using existing OCR results...")
        state["images"] = output
        return state

    # Otherwise attempt to run the async processing synchronously.
    print("🔍 Starting text recognition...")
    
    # Count images for progress display
    img_dir = Path("img")
    image_files = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
    print(f"📊 Found {len(image_files)} images to process")

    images_gen = encode_images("img")
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        # No running loop, create and run one
        print("🚀 Creating new event loop...")
        state["images"] = asyncio.run(process_all_images(images_gen, max_concurrent=15))
        return state

    # If we're in a running event loop (typical in Jupyter), try nest_asyncio to allow run_until_complete.
    if loop.is_running():
        try:
            #print("⚙️ Applying nest_asyncio for Jupyter compatibility...")
            nest_asyncio.apply(loop)
            state["images"] = loop.run_until_complete(process_all_images(images_gen, max_concurrent=15))
            return state
        except Exception as e:
            print(f"❌ Error with nest_asyncio: {e}")
            raise RuntimeError(
                "Cannot run async OCR inside the current event loop. "
                "Either install `nest_asyncio` or run the async cell that awaits `process_all_images(...)` "
                "so that a global `output` variable is available."
            )

    # If event loop not running, run the coroutine directly
    print("🔄 Running coroutine directly...")
    state["images"] = loop.run_until_complete(process_all_images(images_gen, max_concurrent=15))
    return state

def entity_recognition(state: Box) -> Box:
    """Recognize entities in the extracted text."""
    from tqdm import tqdm
    
    print("🏷️ Starting entity recognition...")
    images = state["images"]
    client = OpenAI(base_url=base_url, api_key=api_key)

    for image in tqdm(images, desc="Extracting entities"):
        text = image["markdown_text"]
        prompt = f"Identify and list all named entities (people, places, organizations, dates) mentioned in the following text:\n\n{text}\n\n Be sure to note the context of the entity (i.e. a Person is a witness at a trial, or the driver of a car). Return the entities as a JSON array."
        chat_completion = client.chat.completions.create(
            model=model, #"Qwen/Qwen3-VL-8B-Instruct",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                    ],
                }
            ],
            temperature=0,
            # max_tokens = 100
        )
        response = chat_completion.choices[0].message.content
        try: 
            response = json.loads(response)
        except json.JSONDecodeError:
            print(f"Error decoding JSON for image {image['name']}: {response}")
            
        image["entities"] = response
    state["images"] = images
    return state

def clean_response(state: Box) -> Box:
    """Clean the model response by removing code tags and repeated phrases."""
    from tqdm import tqdm
    
    print("🧹 Cleaning responses...")
    images = state["images"]
    for image in tqdm(images, desc="Cleaning text"):
        text = image["markdown_text"]
        text = remove_code_tags(text)
        text = remove_repeated_phrases(text)
        image["cleaned_text"] = text
        image["image_uri"] = state["file_lookup"].get(image["name"], "")
    state["images"] = images
    return state

def save_state(state: Box) -> Box:
    box_id = (
        state["box_metadata"]
        .get("Identifier", "unknown_box")
        .replace(" ", "_")
        .replace("/", "_")
    )
    # if not output directory, create it
    import os

    os.makedirs("output", exist_ok=True)
    with open(f"output/{box_id}.json", "w") as f:
        json.dump(state, f, indent=4)
    print(f"💾 Saved processed state to output/{box_id}.json")
    return state

In [7]:
from langgraph.graph import StateGraph, START, END


# Build workflow
builder = StateGraph(Box)

# Add nodes
builder.add_node("load_manifest", load_manifest)
builder.add_node("text_recognition", text_recognition)
builder.add_node("entity_recognition", entity_recognition)
builder.add_node("clean_response", clean_response)
builder.add_node("save_state", save_state)

# Add edges to connect nodes
builder.add_edge(START, "load_manifest")
builder.add_edge("load_manifest", "text_recognition")
builder.add_edge("text_recognition", "entity_recognition")
builder.add_edge("entity_recognition", "clean_response")
builder.add_edge("clean_response", "save_state")
builder.add_edge("save_state", END)

# Compile
graph = builder.compile()

In [ ]:
# Invoke

output = graph.invoke({})

Loaded manifest for EAP617/1/3

🔍 Starting text recognition...

📊 Found 207 images to process

Processing images:  12%|█▏        | 24/207 [00:35<10:35,  3.47s/it]

API timeout for image 0084.jpg

...

Processing images:  49%|████▉     | 102/207 [06:49<03:40,  2.10s/it]

API timeout for image 0091.jpg

Processing images:  52%|█████▏    | 108/207 [07:14<07:16,  4.41s/it]

API timeout for image 0063.jpg

Processing images:  53%|█████▎    | 110/207 [07:24<08:08,  5.03s/it]

In [51]:
from rich import print

for item in output["images"][:1]:
    print(item)

{
    'name': '0031.jpg',
    'markdown_text': '```markdown\nGrinular Ld.\nPrinted March 11.\n1893.\n\nH.P.I. Has been ill for several 
months. Complaints of headache spread in the left hypochondriac region. Looks well nourished, rather a rigid 
face.\nV.S. Palpation of arteries marked in neck. Tongue thin, white fur.\nNo typhos, or orthopnea. B.O. once a 
twice daily. Not very eneamic. Congenital rather injured.\n\nL.S. 5ft normal. Lungs (behind) normal.\n\nC.D. as in 
figure.\nA.D. 5th space. K. in. inside N.t.\nSounds normal.\n2nd sound + at L.base.\n\nP. pt. slightly dry. Low 
tension.\n\nDx. Mist Ferr. et Quin.\nMist Mag. Sulph. a.a. 2q. parts.\n3/4 t.d.s.\n\n14. Ld. healed with blue stone
by a dispensary boy. superficial ulcers caused.\n\nMarch 15. Blister on each temple.\n\n17. Mist My. Sulph.\n3/4 
t.d.s.\n\n18. Local Treatment. Yellow colour.\n\n22. Discharged. To continue\nLotic Zinc. q.ii. 2 3/4 t.d.s.\n```',
    'entities': '[\n  {\n    "type": "organization",\n    "value": "Grinular Ld."\n  },\n  {\n    "type": "date",\n
"value": "March 11"\n  },\n  {\n    "type": "date",\n    "value": "1893"\n  },\n  {\n    "type": "date",\n    
"value": "March 15"\n  },\n  {\n    "type": "date",\n    "value": "17"\n  },\n  {\n    "type": "date",\n    
"value": "18"\n  },\n  {\n    "type": "date",\n    "value": "22"\n  },\n  {\n    "type": "person",\n    "value": 
"H.P.I."\n  },\n  {\n    "type": "person",\n    "value": "V.S."\n  },\n  {\n    "type": "person",\n    "value": 
"L.S."\n  },\n  {\n    "type": "person",\n    "value": "C.D."\n  },\n  {\n    "type": "person",\n    "value": 
"A.D."\n  },\n  {\n    "type": "person",\n    "value": "P."\n  },\n  {\n    "type": "person",\n    "value": "Dx."\n
},\n  {\n    "type": "person",\n    "value": "Ld."\n  },\n  {\n    "type": "organization",\n    "value": 
"dispensary boy"\n  }\n]',
    'cleaned_text': 'Grinular Ld.\nPrinted March 11.\n1893.\n\nH.P.I. Has been ill for several months. Complaints 
of headache spread in the left hypochondriac region. Looks well nourished, rather a rigid face.\nV.S. Palpation of 
arteries marked in neck. Tongue thin, white fur.\nNo typhos, orthopnea. B.O. once a twice daily. Not very eneamic. 
Congenital rather injured.\n\nL.S. 5ft normal. Lungs (behind) normal.\n\nC.D. as in figure.\nA.D. 5th space. K. in.
inside N.t.\nSounds normal.\n2nd sound + at L.base.\n\nP. pt. slightly dry. Low tension.\n\nDx. Mist Ferr. et 
Quin.\nMist Mag. Sulph. a.a. 2q. parts.\n3/4 t.d.s.\n\n14. Ld. healed with blue stone by a dispensary boy. 
superficial ulcers caused.\n\nMarch 15. Blister on each temple.\n\n17. Mist My. Sulph.\n3/4 t.d.s.\n\n18. Local 
Treatment. Yellow colour.\n\n22. Discharged. To continue\nLotic Zinc. q.ii. 2 3/4 t.d.s.'
}

## For evaluation section

In [44]:
import srsly 
evaluation_data = srsly.read_json("fmb_evaluation_data.json")

In [45]:
# Get manifest URIs from evaluation data
manifest_uris = [entry["manifest_uri"] for entry in evaluation_data.values()]
len(manifest_uris)

37

In [65]:
from pathlib import Path
for manifest in manifest_uris[9:10]:
    print(f"Processing manifest: {manifest}")
    if Path("output/" + manifest.split("/")[-2].replace("-", "_") + ".json").exists():
        print(f"Skipping {manifest}, already processed.")
        continue
    output = graph.invoke(
        {
            "box_manifest_uri": manifest,
            "start_index": 0,
            #end_index": 3,
            "image_width": 400,
        }
    )

Processing manifest: https://eap.bl.uk/archive-file/EAP1477-1-1-10/manifest

Successfully fetched manifest for Caja 10

Found 2011 images in manifest

Extracting text from images:   0%|          | 6/2011 [00:19<1:51:49,  3.35s/it]

Request error: The read operation timed out

...

Request error: The read operation timed out

Extracting text from images:   0%|          | 10/2011 [00:48<3:39:32,  6.58s/it]

Request error: The read operation timed out

Extracting text from images:   1%|          | 11/2011 [01:00<3:03:07,  5.49s/it]


KeyboardInterrupt: 